Currently this does not pass `loadable_vars` explictly to `vz.open_virtual_mfdataset()`. It is relying on the internal functionality of virtualizarr to identify the 1D coordinate variables that should be passed to `loadable_vars`.  

For reference use Luis' notebook, which also demonstrates how to create VDS references using HTTPS access: https://notebooksharing.space/view/a0214376374af43776a54bf69b0b0308bbc0499fad345fcb5eb70e9ad9cf46c6#displayOptions=

In [3]:
import earthaccess as ea
import xarray as xr
import zarr
from urllib.parse import urlparse
import fsspec
import warnings
import logging

import dask
from dask.distributed import Client, LocalCluster
from coiled import Cluster as CoiledCluster
import numpy as np

import icechunk
import obstore
from obstore.auth.earthdata import NasaEarthdataCredentialProvider # Only S3 links access in-region
from obstore.store import HTTPStore, S3Store
from virtualizarr.parsers import DMRPPParser, NetCDF3Parser, HDFParser
from  obspec_utils.registry import ObjectStoreRegistry
import virtualizarr as vz

for _ in (ea, vz, zarr, xr, obstore, dask, fsspec, icechunk):
    print(f"{_.__name__}: {_.__version__}")

earthaccess: 0.18.0
virtualizarr: 2.7.3
zarr: 3.2.1
xarray: 2026.7.0
obstore: 0.9.5
dask: 2024.5.2
fsspec: 2026.3.0
icechunk: 2.1.2


In [4]:
auth = ea.login()

Enter your Earthdata Login username:  edward.m.armstrong
Enter your Earthdata password:  ········


## 1. Create Dask cluster

In [5]:
cloud_opts = {
    "region": "us-west-2",
    "worker_vm_types": ["t3a.medium"],
    "spot_policy": "spot_with_fallback",
    # "arm": False,
    "name": "test-vd",
    "environ": {"EARTHDATA_TOKEN": auth.token["access_token"]}
}

def create_dask_cluster(environment="local",
                        n_workers=8,
                        cloud_opts={}):
    if environment=="local":
        import logging
        print("Creating new local Dask client")
        cluster = LocalCluster(
            n_workers=n_workers,
            threads_per_worker=1,
            silence_logs=logging.ERROR)
    else:
        print("Creating new Coiled Dask client")
        cluster = CoiledCluster(n_workers=n_workers,
                                **cloud_opts)    

    client = Client(cluster)
    return (client, cluster)

In [6]:
import multiprocessing

# Check how many cpu's are on this VM:
print("CPU count =", multiprocessing.cpu_count())

CPU count = 8


In [7]:
if "client" not in locals() or "cluster" not in locals():
    client, cluster = create_dask_cluster(
        environment="local",
        n_workers=8,
        cloud_opts=cloud_opts)
client

Creating new local Dask client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-eyglk.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-eyglk.dask.host/jupyter/proxy/8787/status,Workers: 8
Total threads: 8,Total memory: 29.41 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38345,Workers: 8
Dashboard: https://cluster-eyglk.dask.host/jupyter/proxy/8787/status,Total threads: 8
Started: Just now,Total memory: 29.41 GiB
Comm: tcp://127.0.0.1:37861,Total threads: 1
Dashboard: https://cluster-eyglk.dask.host/jupyter/proxy/34913/status,Memory: 3.68 GiB
Nanny: tcp://127.0.0.1:34311,


#### Following code is needed to give EDL creds explicitly to each Dask worker

In [8]:
def silence_worker_warnings_and_auth(token):
    import warnings
    import logging
    # local instance of earthaccess is explicitely authenticated
    import earthaccess as ea
    import os

    if token:
        os.environ["EARTHDATA_TOKEN"]= token
        auth = ea.login(strategy="environment")
    
    warnings.filterwarnings("ignore")
    for name in ["distributed", "xarray", "py.warnings", "fsspec", "h5netcdf", "h5py"]:
        logging.getLogger(name).setLevel(logging.ERROR)

In [9]:
%%capture
client.run(silence_worker_warnings_and_auth, auth.token["access_token"])

## 2. Create the VDS

## 2.1 Create combined VDS ref files for each collection

In [10]:
shortnames = [
    "OSTIA-UKMO-L4-GLOB-REP-v2.0"
]

In [11]:
%%time

vds_s3_list = []

for sn in shortnames:
    
    print(sn)

    auth = ea.login()
    
    ## 1. Get granule metadata and s3 links ------------------------------------------------
    results = ea.search_data(
        short_name=sn,
        provider="POCLOUD",
        cloud_hosted=True,
        temporal=("1990-01-01", "1999-12-31")       
    )
        # S3 links
    granule_data_urls_s3 = [
        granule.data_links(access="direct")[0] for granule in results
    ]
    print("Number of file found: ", len(results))

    
    ## 2. Setup store registry -------------------------------------------------------------
    access = "direct"
    credentials_endpoint = "https://archive.podaac.earthdata.nasa.gov/s3credentials"
        # Get bucket location of granules from the first granule's metadata:
    parsed_url = urlparse(results[0].data_links(access=access)[0])
    bucket = parsed_url.netloc
    
    s3_store = S3Store(
        bucket=bucket,
        region="us-west-2",
        credential_provider=NasaEarthdataCredentialProvider(credentials_endpoint, auth=auth.token["access_token"]),
        virtual_hosted_style_request=False,
        client_options={"allow_http": True},
    )
    s3_obstore_registry = ObjectStoreRegistry({f"s3://{bucket}": s3_store})


    ## 3. Setup any data preprocessing and kwargs to pass to xr.combine_nested() -------------
        # not doing any preprocessing currently:
    def preprocess_func(ds):
        return ds
    
    xr_combine_nested_kwargs = {
        "concat_dim": "time",  # Concatenate files along the time dimension
        "preprocess": preprocess_func, # Normalize the dataframe 
        "data_vars": "minimal",  # Only load data variables that include the concat_dim
        "coords": "minimal",  # Only load coordinate variables that include the concat_dim
        "compat": "override",  # Avoid coordinate conflicts by picking the first
        "combine_attrs": "override",  # Avoid attribute conflicts by picking the first
    }


    ## 4. Create VDS reference - does the single file references and concatentation
    ## in one function ------------------------------------------------------------------------
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message="Numcodecs codecs*",
            category=UserWarning,
        )
        group = None
        vds_s3 = vz.open_virtual_mfdataset(
            urls=granule_data_urls_s3,
            registry=s3_obstore_registry,
            parser=HDFParser(),
            decode_times=False,
            parallel="dask",
            combine="nested",
            **xr_combine_nested_kwargs,
        )

    vds_s3_list.append(vds_s3)

OSTIA-UKMO-L4-GLOB-REP-v2.0


/opt/coiled/env/lib/python3.12/site-packages/earthaccess/results.py:348: FutureWarning: As of version 1.0, `DataGranule.size` will be accessed as an attribute; e.g. use `DataCollection.size` **not** `DataCollection.size()`
  self["size"] = self.size()


Number of file found:  3653
CPU times: user 46.8 s, sys: 8.73 s, total: 55.5 s
Wall time: 8min 25s


## 2.2 Multi-collection VDS

In [12]:
## Combine all into single VDS:
vds_composite_s3 = xr.combine_nested(
    vds_s3_list, concat_dim=None, 
    compat='override', combine_attrs='drop_conflicts'
)

In [13]:
print(vds_composite_s3)

<xarray.Dataset> Size: 568GB
Dimensions:           (time: 3653, lat: 3600, lon: 7200)
Coordinates:
  * time              (time) int32 15kB 283953600 284040000 ... 599486400
  * lat               (lat) float32 14kB -89.97 -89.93 -89.88 ... 89.93 89.97
  * lon               (lon) float32 29kB -180.0 -179.9 -179.9 ... 179.9 180.0
Data variables:
    analysed_sst      (time, lat, lon) int16 189GB ManifestArray<shape=(3653,...
    analysis_error    (time, lat, lon) int16 189GB ManifestArray<shape=(3653,...
    sea_ice_fraction  (time, lat, lon) int8 95GB ManifestArray<shape=(3653, 3...
    mask              (time, lat, lon) int8 95GB ManifestArray<shape=(3653, 3...
Attributes: (12/47)
    Conventions:                CF-1.4, ACDD-1.3
    title:                      Global SST & Sea Ice Analysis, L4 OSTIA, 0.05...
    summary:                    A merged, multi-sensor L4 Foundation SST product
    references:                 Donlon, C.J., Martin, M., Stark, J.D., Robert...
    institution:   

In [14]:
## Modify/add some attributes:
vds_composite_s3.attrs['time_coverage_start'] = '1990-01-01T00:00:00Z'
vds_composite_s3.attrs['time_coverage_end'] = '1999-12-31T00:00:00Z'
vds_composite_s3.attrs['identifier_product_doi'] = "https://doi.org/10.5067/GHOST-4RM02"
vds_composite_s3.attrs['date_created'] = "2026-08-05T00:00:00Z"
vds_composite_s3.attrs['history'] = "Icechunk v2  VDS for OSTIA"


## 2.3 Duplicate to create a VDS with HTTP endpoints

In [15]:
from pathlib import Path

In [16]:
def s3_to_http_url(old_s3_path: str) -> str:
    """
    Change S3 endpoint to HTTP.
    """
    from pathlib import Path

    https_base = "https://archive.podaac.earthdata.nasa.gov/"
    return str(https_base + old_s3_path.split("//")[-1])

In [17]:
vds_composite_http = vds_composite_s3.vz.rename_paths(s3_to_http_url)

In [18]:
print(vds_composite_http)

<xarray.Dataset> Size: 568GB
Dimensions:           (time: 3653, lat: 3600, lon: 7200)
Coordinates:
  * time              (time) int32 15kB 283953600 284040000 ... 599486400
  * lat               (lat) float32 14kB -89.97 -89.93 -89.88 ... 89.93 89.97
  * lon               (lon) float32 29kB -180.0 -179.9 -179.9 ... 179.9 180.0
Data variables:
    analysed_sst      (time, lat, lon) int16 189GB ManifestArray<shape=(3653,...
    analysis_error    (time, lat, lon) int16 189GB ManifestArray<shape=(3653,...
    sea_ice_fraction  (time, lat, lon) int8 95GB ManifestArray<shape=(3653, 3...
    mask              (time, lat, lon) int8 95GB ManifestArray<shape=(3653, 3...
Attributes: (12/48)
    Conventions:                CF-1.4, ACDD-1.3
    title:                      Global SST & Sea Ice Analysis, L4 OSTIA, 0.05...
    summary:                    A merged, multi-sensor L4 Foundation SST product
    references:                 Donlon, C.J., Martin, M., Stark, J.D., Robert...
    institution:   

## 2.4 Save to Icechunk stores

One store each for the S3 and HTTP endpoints.

In [19]:
## Https store 

def create_local_icechunk_repo_httpaccess(repo_name: str, vcc_http_base: str):
    """
    Creates an icechunk repository locally, with pointers to data chunks in HTTPS locations.

    Inputs
    ------
    * repo_name (str): path to icechunk repo.
    """
    storage = icechunk.local_filesystem_storage(path=repo_name)
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(
            vcc_http_base,
            icechunk.http_store()
        )
    )
    return icechunk.Repository.create(storage, config)

In [20]:
## S3 store

def create_local_icechunk_repo_s3access(repo_name: str, vcc_bucket: str):
    """
    Creates an icechunk repository locally, with pointers to data chunks in S3 locations.

    Inputs
    ------
    * repo_name (str): path to icechunk repo.
    """
    storage = icechunk.local_filesystem_storage(path=repo_name)
    config = icechunk.RepositoryConfig.default()
    config.set_virtual_chunk_container(
        icechunk.VirtualChunkContainer(
            url_prefix=vcc_bucket+"/",
            store=icechunk.s3_store(region="us-west-2", anonymous=True)
        )
    )
    return icechunk.Repository.create(storage, config)

#### S3 endpoints

In [23]:
# Create icechunk repo and session
fname_store_s3access = "OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3"
repo = create_local_icechunk_repo_s3access(fname_store_s3access, "s3://"+bucket)
session = repo.writable_session("main")

  2026-08-05T18:37:33.004402Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329



In [24]:
# Then write to the store:
vds_composite_s3.virtualize.to_icechunk(session.store)
session.commit("Initial commit.")

'F0V7A3GJ5KJA9VA02K6G'

#### HTTP endpoints

In [28]:
# Create icechunk repo and session
fname_store_httpaccess = "OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https"
repo = create_local_icechunk_repo_httpaccess(fname_store_httpaccess, "https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/")
session = repo.writable_session("main")

  2026-08-05T18:44:59.272626Z  WARN icechunk_arrow_object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk-arrow-object-store/src/lib.rs:329



In [29]:
# Then write to the store:
vds_composite_http.virtualize.to_icechunk(session.store)
session.commit("Initial commit.")

'JSGRCAGW0RJQH8S4NPHG'

In [27]:
!tar -cvf $fname_store_s3access".tar" $fname_store_s3access

OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/chunks/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/chunks/RC73ZQNEFWQVNZHFM7AG
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/chunks/886YWPK0Q3C38PTFYZH0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/ZQ2F1F4ZCCFVAMT8EQH0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/C5EZM86VKBAVXJXSXHFG
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/3EA73KBN8PP998XFE8X0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/N3QPQG0H7YZCMB323QWG
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/5SGVV64XNKPT05FVJ0M0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/TWA4BMEYPNRPVM6ZYGH0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/manifests/TCC4KDK8XA4F5KX72EN0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/overwritten/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.s3/overwritten/repo.30717724941580.06WS1AD1DDX8YF56DWSG
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2

In [30]:
!tar -cvf $fname_store_httpaccess".tar" $fname_store_httpaccess

OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/chunks/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/chunks/00X0TATS8H6TY7S578RG
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/chunks/HGDBSZ52R3MP2NNZWPFG
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/THC657RNMFR2XEH100H0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/14FZ6DEY9PAYH4BPGAZ0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/ZGJV54TNT0M2K323M3Z0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/9H9MAA79FJ8GE7THH3D0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/DA18K72DYNCTT17RTW20
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/FMAS4D3BSR9ZVDZNTRJG
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/manifests/YPPG1MEKBFBRDEKYWYF0
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/overwritten/
OSTIA-UKMO-L4-GLOB-REP-v2.0.icechunk_v2.https/overwritten/repo.30717724477639.V7KA4H18ZXKDYVT9QD